In [4]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import european_analytics
from models import mSABR,SABR
from scipy.stats import lognorm
from matplotlib.lines import Line2D
from stats import PathStatistics

In [ ]:
# Model parameters
F0=0.05
lambda_=0.04
F_s=F0+lambda_
beta=0.5
nu=0.5
rho=-0.3
sigma_atm=0.01
T = 2.0

# Simulation parameters
N_PATHS = 40_000
SEED    = 42

# Mean reversion grid: none / moderate / strong
kappas = [0.0, 1.0, 4.0]
labels = ['no MR (kappa=0)', 'moderate (kappa=1)', 'strong (kappa=4)']
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(kappas)))

def build(kappa, T, n_paths=N_PATHS):
    an = european_analytics.EuropeanAnalyticsMRSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_, kappa)
    sim = mSABR(F0=F_s, A0=an.alpha, rho=rho, T=T,n_steps=int(252*T), n_paths=n_paths,beta=beta, nu=nu, kappa=kappa, theta=an.alpha, seed=SEED)
    return an, sim

# kappa=0 must reproduce standard SABR
ref = european_analytics.EuropeanAnalyticsSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_)
chk = european_analytics.EuropeanAnalyticsMRSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_, 0.0)
print(f"kappa=0 reduces to standard SABR: {np.isclose(ref.alpha, chk.alpha)}")
print(f"sigma_ln_atm: {chk.sigma_ln_atm:.6f}")
print(f"alpha: {chk.alpha:.6f}")

In [ ]:
# Terminal distribution of F(T) across mean reversion (kappa sweep)
# Fix everything, sweep kappa = [0, 1, 4]; alpha calibrated per-kappa so ATM stays pinned.
N_STEPS = 1000

fig, axes = plt.subplots(1, len(kappas), figsize=(16, 4.5), sharex=True, sharey=True)

# lognormal reference (flat-vol Black at the calibrated ATM lognormal vol)
sigma_ln_atm = chk.sigma_ln_atm
mu_ln    = np.log(F_s) - 0.5 * sigma_ln_atm**2 * T
sigma_ln = sigma_ln_atm * np.sqrt(T)

rows = []
xlim = None
for ax, kap, lab, col in zip(axes, kappas, labels, colors):
    an = european_analytics.EuropeanAnalyticsMRSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_, kap)
    sim = mSABR(F0=F_s, A0=an.alpha, rho=rho, T=T, n_steps=N_STEPS, n_paths=N_PATHS,
                beta=beta, nu=nu, kappa=kap, theta=an.alpha, seed=SEED)
    F_T = sim.run()[0][:, -1] - lambda_

    if xlim is None:  # fix the window from the kappa=0 (widest) case
        xlim = np.percentile(F_T, [0.3, 99.7])
        x_ref = np.linspace(xlim[0], xlim[1], 400)
        ln_pdf = lognorm.pdf(x_ref + lambda_, s=sigma_ln, scale=np.exp(mu_ln))

    ax.hist(F_T, bins=np.linspace(xlim[0], xlim[1], 120), density=True, color=col, alpha=0.75)
    ax.plot(x_ref, ln_pdf, 'k--', lw=1.5, label='Lognormal ref')
    ax.axvline(F0, color='gray', lw=1, ls='--', label='$F_0$')
    ax.set_title(f'{lab}\n(nu_eff={an.nu_eff:.3f}, rho_eff={an.rho_eff:+.3f})')
    ax.set_xlabel(r'$F_T$'); ax.set_ylabel('Density')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    p_lo, p_hi = PathStatistics.quantile(F_T, [0.5, 99.5])
    rows.append((kap, an.nu_eff, an.rho_eff, PathStatistics.std_dev(F_T), PathStatistics.skewness(F_T),
                 PathStatistics.kurtosis(F_T), p_lo, p_hi))

axes[0].set_xlim(*xlim)
fig.suptitle(f'Terminal distribution across mean reversion  (T={T}y, beta={beta}, nu={nu}, rho={rho})',
             y=1.03)
plt.tight_layout()
plt.show()

print(f"{'kappa':>6}{'nu_eff':>9}{'rho_eff':>9}{'std':>10}{'skew':>9}{'kurt':>9}{'P0.5':>10}{'P99.5':>10}")
for kap, nu_eff, rho_eff, s, sk, ku, plo, phi in rows:
    print(f"{kap:6.1f}{nu_eff:9.3f}{rho_eff:+9.3f}{s:10.3f}{sk:+9.3f}{ku:+9.3f}{plo:10.5f}{phi:10.5f}")

## Terminal Distribution: SABR vs Mean-Reverting SABR ($\kappa$ Sweep)

**Experiment:** We fix $T = 2\text{y}$, $\beta = 0.5$, $\nu = 0.5$, $\rho = -0.3$ and sweep the mean
reversion parameter $\kappa \in \{0, 1, 4\}$ (none / moderate / strong). $\kappa = 0$ is basically standard
SABR. The vol process is $dA = \kappa(\theta - A)\,dt + \nu A\,dW_2$ with $\theta = A_0 = \alpha$, so
the vol starts at the same level in every case; the only thing changing across panels
is the $\kappa$. Also, $\alpha$ is recalibrated (via the mrSABR library) for each case so all three cases share the same ATM vol, which means any difference in the terminal distribution of $F_T$ is entirely due to $\kappa$.

**Effective vol-of-vol:** The parameter driving the shape of terminal distribution $\nu_{\text{eff}}$,
falls from $0.500$ to $0.269$ to $0.098$ as $\kappa$ increases. Mean reversion acts, as a suppressor of the effective vol-of-vol, the harder the pull-back, the less the vol is able to wander over the life of the option.

**Kurtosis:** Excess kurtosis drops from $+2.376$ at $\kappa = 0$ to $+0.053$ at $\kappa = 4$, from
fat-tailed back to lognormal-like. In standard SABR the vol has no drift, so its variance keeps
growing and some paths reach very high vol, pushing $F_T$ to extreme values on both sides. Mean
reversion pulls $A_t$ back towards $\theta$, so the vol stays bounded and the tails of $F_T$ thin
out. At $\kappa = 4$ the distribution closely matches with the lognormal reference, while
at $\kappa = 0$ it sits above it in both wings, and the 0.5–99.5% range tightens from
$[-0.0008,\ 0.0948]$ to $[0.0147,\ 0.0886]$ as $\kappa = 0$ increases. Also noteworthy: The $\kappa = 0$ left tail even reaches negative rates,
which mean reversion removes.

**Skew:** The skew moves from $-0.268$ to $-0.104$ to $+0.1$, so the negative skew is removed and
even flips sign. With $\rho = -0.3$, downward moves in $F$ come with upward moves in vol, which
builds a fat left tail. But this only works when the vol actually moves. As $\kappa$ grows,
$\nu_{\text{eff}}$ shrinks and $A_t$ stays near $\theta$, so $\rho$ has little to act on and the
negative skew fades. The small positive skew left at $\kappa = 4$ is just the natural $\beta = 0.5$
lognormal skew, which was hidden underneath the negative $\rho$ effect before.


In [ ]:
# Verify effective parameter mapping
kap_test = 4.0

# mrSABR with original params
an_mr = european_analytics.EuropeanAnalyticsMRSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_, kap_test)

sim_mr = mSABR(F0=F_s, A0=an_mr.alpha, rho=rho, T=T,n_steps=N_STEPS, n_paths=N_PATHS,beta=beta, nu=nu, kappa=kap_test, theta=an_mr.alpha, seed=SEED)
F_T_mr = sim_mr.run()[0][:, -1] - lambda_

# Standard SABR with effective params
an_std = european_analytics.EuropeanAnalyticsSABR(F_s, T, beta, an_mr.nu_eff, an_mr.rho_eff, sigma_atm, lambda_)

sim_std = SABR(F0=F_s, A0=an_std.alpha, rho=an_mr.rho_eff, T=T,n_steps=N_STEPS, n_paths=N_PATHS,beta=beta, nu=an_mr.nu_eff, seed=SEED)
F_T_std = sim_std.run()[0][:, -1] - lambda_

# Overlay
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(min(F_T_mr.min(), F_T_std.min()),max(F_T_mr.max(), F_T_std.max()),120)

ax.hist(F_T_mr,  bins=bins, density=True, alpha=0.6, color='blue',label=f'mrSABR (κ={kap_test}, nu={nu}, rho={rho})')
ax.hist(F_T_std, bins=bins, density=True, alpha=0.5, color='red',label=f'Std SABR (nu_eff={an_mr.nu_eff:.3f}, rho_eff={an_mr.rho_eff:+.3f})')
ax.set_title(f'Effective parameter verification: κ={kap_test}')
ax.set_xlabel(r'$F_T$'); ax.set_ylabel('Density')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Stats comparison
print(f"{'':>15}{'mrSABR':>12}{'Std SABR(eff)':>14}{'diff%':>9}")
for name, fn in [('mean', np.mean), ('std', np.std),
                 ('skew', PathStatistics.skewness), ('kurtosis', PathStatistics.kurtosis)]:
    v1, v2 = fn(F_T_mr), fn(F_T_std)
    pct = (v2 - v1) / abs(v1) * 100 if abs(v1) > 1e-10 else 0
    print(f"{name:>15}{v1:12.5f}{v2:14.5f}{pct:+8.1f}%")

### Experiment: Effective Parameter Verification

**Test:** Simulate standard SABR with ($\nu_{eff}$=0.098, $\rho_{eff}$=−0.336) and compare againstmrSABR ($\kappa$=4, $\nu$=0.5, $\rho$=−0.3). If the effective parameter mapping works, the two terminal distributions should match.

**Results:** Mean, std, and skew match within 1% of each other, thus the effective parameter approximation captures the first three moments well. Kurtosis shows a 23% gap:
the mrSABR produces slightly fatter tails (0.053 vs 0.041) because the time-varying vol-of-vol path in the mrSABR creates transient bursts of high volatility that a constant $\nu_{eff}$ cannot replicate. This is the expected limitation of the $O(\epsilon^2)$ approximation — it matches the leading-order smile structure but not the tail behavior.

The histogram overlay confirms visually: the bulk of the distribution is nearly identical.

In [ ]:
# Does mean reversion matter more at short or long maturity
# Fix kappa; sweep T
maturities = [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
kap_no, kap_mr = 0.0, 2.0

kurt_no, kurt_mr = [], []
for T_i in maturities:
    n_steps_i = min(max(int(500 * T_i), 200), 5000)
    for kap, kurt_list in [(kap_no, kurt_no), (kap_mr, kurt_mr)]:
        an  = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_i, beta, nu, rho, sigma_atm, lambda_, kap)
        sim = mSABR(F0=F_s, A0=an.alpha, rho=rho, T=T_i, n_steps=n_steps_i, n_paths=N_PATHS,
                    beta=beta, nu=nu, kappa=kap, theta=an.alpha, seed=SEED)
        F_T = sim.run()[0][:, -1] - lambda_
        kurt_list.append(PathStatistics.kurtosis(F_T))

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(maturities, np.maximum(kurt_no, 1e-2), 'o-', color='red', lw=2, label='no MR (kappa=0)')
ax.plot(maturities, np.maximum(kurt_mr, 1e-2), 's-', color='green', lw=2, label='MR (kappa=2)')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Excess kurtosis of $F_T$')
ax.set_title(f'Mean reversion effect for diff maturity maturity  (beta={beta}, nu={nu}, rho={rho})')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"{'T':>6}{'kurt(noMR)':>13}{'kurt(MR)':>11}{'kappa*T':>10}")
for i, T_i in enumerate(maturities):
    print(f"{T_i:6.2f}{kurt_no[i]:13.2f}{kurt_mr[i]:11.2f}{kap_mr*T_i:10.2f}")

## Mean Reversion Effect vs Maturity

**Experiment:** We fix $\beta = 0.5$, $\nu = 0.5$, $\rho = -0.3$ and compare no mean reversion ($\kappa = 0$) against mean reversion ($\kappa = 2$), sweeping the maturity $T$ across $[0.25, 0.5, 1, 2, 5, 10]$ years. At each $T$ we measure the excess kurtosis of $F_T$. The goal is to see whether mean reversion matters more at short or long maturity. All runs share the same seed, and $\alpha$ is recalibrated at each $T$ so the ATM level stays pinned.

**Result:** At short maturities the two models are almost the same. At $T = 0.25$ the kurtosis is $0.24$ without mean reversion and $0.16$ with it which is barely any difference. As $T$ grows the gap increases. Without mean reversion the kurtosis keeps increasing (from $0.24$ up to $51.8$ at $T = 10$), while with mean reversion it stays almost constant, around $0.1$–$0.2$ at every maturity.

**Conclusion:** Without mean reversion the vol keeps wandering, and the longer the option lives, the more time it has to drift to extreme levels, so the tails of $F_T$ get fatter and fatter with $T$. Mean reversion keeps pulling the vol back, so no matter how long the maturity, the vol stays bounded and the tails do not blow up. This is why the no-MR kurtosis grows without limit while the MR kurtosis stays flat.

In [ ]:
# Effect of mean reversion by strike
# T fixed at 5 years; sweep kappa in [0,1,4]
T_long = 5.0

an0 = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_long, beta, nu, rho, sigma_atm, lambda_, 0.0)
std_sm = an0.sigma_ln_atm * F_s * np.sqrt(T_long)

# log moneyness
logm_max = 2.5 * std_sm / F_s
logm = np.linspace(-logm_max, logm_max, 31)
K_grid = F_s * np.exp(logm)
moneyness = logm

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

smiles = {}
for kap, lab, col in zip(kappas, labels, colors):
    an, sim = build(kap, T_long)
    F_s_T = sim.run()[0][:, -1]
    v_hagan = an.implied_vol_smile(K_grid)
    v_mc    = an.implied_vol_smile_mc(F_s_T, K_grid)
    smiles[kap] = v_hagan
    axes[0].plot((K_grid - lambda_)*100, v_hagan*100, color=col, lw=2, label=lab)
    axes[0].plot((K_grid - lambda_)*100, v_mc*100, 'o', color=col, ms=3, alpha=0.55)

handles, _ = axes[0].get_legend_handles_labels()
handles.append(Line2D([0],[0], marker='o', ls='', color='gray', ms=4, alpha=0.6, label='MC'))
axes[0].axvline(F0*100, color='gray', lw=1, ls='--')
axes[0].set_xlabel('Strike (%)'); axes[0].set_ylabel('Implied vol (%)')
axes[0].set_title(f'Smile across kappa (T={T_long}y)')
axes[0].legend(handles=handles, fontsize=8); axes[0].grid(alpha=0.3)

# how much each strike moves relative to no-MR
base = smiles[0.0]
for kap, lab, col in zip(kappas[1:], labels[1:], colors[1:]):
    axes[1].plot(moneyness, (smiles[kap] - base)*100, color=col, lw=2, label=lab)
axes[1].axhline(0, color='gray', lw=1, ls='--')
axes[1].axvline(0, color='gray', lw=1, ls='--', label='ATM')
axes[1].set_xlabel('Log-moneyness  log(K / F)')
axes[1].set_ylabel('Implied vol change vs no-MR (vol %)')
axes[1].set_title(f'Effect of mean reversion by strike (T={T_long}y)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

fig.suptitle(f'ATM vs wings  (T={T_long}y, beta={beta}, nu={nu}, rho={rho})', y=1.02)
plt.tight_layout()
plt.show()

atm_idx = np.argmin(np.abs(K_grid - F_s))
print(f"{'kappa':>6}{'ATM change':>12}{'left wing':>14}{'right wing':>14}")
for kap in kappas:
    d = (smiles[kap] - base) * 100
    print(f"{kap:6.1f}{d[atm_idx]:12.3f}{d[0]:14.3f}{d[-1]:14.3f}")

## ATM vs Wings: Which Strikes Does Mean Reversion Affect Most?

**Experiment:** We fix $T = 5\text{y}$ (long enough for mean reversion to have effect), $\beta = 0.5$, $\nu = 0.5$, $\rho = -0.3$, and compare the smile across
$\kappa \in \{0, 1, 4\}$. The left panel shows the smiles (lines = Hagan, dots = MC simulation); the right panel shows how much each strike's implied vol *moves* relative to the no-MR case, plotted against moneyness.

**Result of mean reversion:** The ATM vol does not change at all for every $\kappa$. All the change
happens in the wings, and it increases the further out you go. At low-strikes the vol drops by
$6.86$ points ($\kappa = 1$) and $8.4$ points ($\kappa = 4$), while ATM vol is unchanged. In the left
panel this shows up as three smiles coinciding together at the money and spreading apart at the edges, with
the flatter smiles corresponding to stronger mean reversion.

**Left wing moves more than the right:** The drop is much bigger on the low-strike side than the high-strike side. This is due to the negative $\rho = -0.3$ the left wing was the steepest, most tail-heavy part of the smile to begin with, so it has the most reduction when mean reversion thins the tails. As you can see in teh right panel - the curve dips deeper on the left of ATM than the right.

**Conclusion:** Mean reversion affects the wings, not ATM. It barely touches at-the-money options and matters most for deep OTM/ITM strikes, especially on the downside(low strikes). This makes sense as we say in the first experiment that mean reversion thins the tails of $F_T$, and the tails are exactly what price the wings of the smile, while ATM is set by the bulk of the distribution, which hardly changes.

In [ ]:
# flip rho to positive, expect to see more drop on the rigth side of the x-axis(moneyness)
# same as last experiment but rho = +0.3 instead of -0.3
rho_pos = +0.3

an0_p = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_long, beta, nu, rho_pos, sigma_atm, lambda_, 0.0)
std_p = an0_p.sigma_ln_atm * F_s * np.sqrt(T_long)

logm_max = 2.5 * std_p / F_s
logm_p = np.linspace(-logm_max, logm_max, 31)
K_grid_p = F_s * np.exp(logm_p)
moneyness_p = logm_p

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

smiles_p = {}
for kap, lab, col in zip(kappas, labels, colors):
    an = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_long, beta, nu, rho_pos, sigma_atm, lambda_, kap)
    sim = mSABR(F0=F_s, A0=an.alpha, rho=rho_pos, T=T_long, n_steps=int(252*T_long),
                n_paths=N_PATHS, beta=beta, nu=nu, kappa=kap, theta=an.alpha, seed=SEED)
    F_s_T = sim.run()[0][:, -1]
    v_hagan = an.implied_vol_smile(K_grid_p)
    v_mc    = an.implied_vol_smile_mc(F_s_T, K_grid_p)
    smiles_p[kap] = v_hagan
    axes[0].plot((K_grid_p - lambda_)*100, v_hagan*100, color=col, lw=2, label=lab)
    axes[0].plot((K_grid_p - lambda_)*100, v_mc*100, 'o', color=col, ms=3, alpha=0.55)

handles, _ = axes[0].get_legend_handles_labels()
handles.append(Line2D([0],[0], marker='o', ls='', color='gray', ms=4, alpha=0.6, label='MC'))
axes[0].axvline(F0*100, color='gray', lw=1, ls='--')
axes[0].set_xlabel('Strike (%)'); axes[0].set_ylabel('Implied vol (%)')
axes[0].set_title(f'Smile across kappa')
axes[0].legend(handles=handles, fontsize=8); axes[0].grid(alpha=0.3)

base_p = smiles_p[0.0]
for kap, lab, col in zip(kappas[1:], labels[1:], colors[1:]):
    axes[1].plot(moneyness_p, (smiles_p[kap] - base_p)*100, color=col, lw=2, label=lab)
axes[1].axhline(0, color='gray', lw=1, ls='--')
axes[1].axvline(0, color='gray', lw=1, ls='--', label='ATM')
axes[1].set_xlabel('Log-moneyness  log(K / F)')
axes[1].set_ylabel('Implied vol change vs no-MR (vol %)')
axes[1].set_title(f'Effect of mean reversion by strike')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

fig.suptitle(f'ATM vs wings, positive rho  (T={T_long}y, beta={beta}, nu={nu}, rho=+{rho_pos})', y=1.02)
plt.tight_layout()
plt.show()

atm_idx_p = np.argmin(np.abs(K_grid_p - F_s))
print(f"rho = +{rho_pos}")
print(f"{'kappa':>6}{'ATM change':>12}{'left wing':>14}{'right wing':>14}")
for kap in kappas:
    d = (smiles_p[kap] - base_p) * 100
    print(f"{kap:6.1f}{d[atm_idx_p]:14.3f}{d[0]:14.3f}{d[-1]:14.3f}")

In [ ]:
# Experiment: does kappa add a genuine degree of freedom, or is it just repackaging nu
kappas_test = [0.0, 1.0, 2.0, 4.0, 8.0]
colors_test = plt.cm.viridis(np.linspace(0.1, 0.9, len(kappas_test)))
logm_max = 0.8
K_grid = F_s * np.exp(np.linspace(-logm_max, logm_max, 31))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
rows = []
for kap, col in zip(kappas_test, colors_test):
    an = european_analytics.EuropeanAnalyticsMRSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_, kap)

    sim = mSABR(F0=F_s, A0=an.alpha, rho=rho, T=T,n_steps=N_STEPS, n_paths=N_PATHS,beta=beta, nu=nu, kappa=kap, theta=an.alpha, seed=SEED)
    F_paths = sim.run()[0]
    F_shifted = F_paths[:, -1]
    F_T = F_shifted - lambda_

    mc_vols   = an.implied_vol_smile_mc(F_shifted, K_grid)
    hagan_vols = an.implied_vol_smile(K_grid)
    residual  = (mc_vols - hagan_vols) * 1e4

    lab = f'κ={kap:.0f} (nu_eff={an.nu_eff:.3f}, rho_eff={an.rho_eff:+.3f})'

    # MC smiles
    axes[0].plot(K_grid, mc_vols, '-', color=col, lw=1.8, label=lab)
    axes[0].plot(K_grid, hagan_vols, '--', color=col, lw=1, alpha=0.6)

    # residuals (MC − Hagan) in bps
    axes[1].plot(K_grid, residual, '-', color=col, lw=1.8, label=f'κ={kap:.0f}')

    # Max absolute residual
    max_res = np.nanmax(np.abs(residual))
    rows.append((kap, an.nu_eff, an.rho_eff, max_res))

axes[0].set_title('MC smile (solid) vs Hagan effective (dashed)')
axes[0].set_xlabel('Strike'); axes[0].set_ylabel('Implied vol')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Residual: MC - Hagan(nu_eff, rho_eff)')
axes[1].set_xlabel('Strike'); axes[1].set_ylabel('Residual (bps)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

fig.suptitle(f'Experiment: nu x k interaction  (nu={nu}, T={T}, beta={beta}, rho={rho})', y=1.02)
plt.tight_layout(); plt.show()

print(f"{'kappa':>6}{'nu_eff':>9}{'rho_eff':>9}{'max_resid_bps':>15}")
for kap, ne, re, mr in rows:
    print(f"{kap:6.1f}{ne:9.3f}{re:+9.3f}{mr:15.1f}")

### Experiment : does kappa add a genuine degree of freedom, or is it just repackaging nu

**What we see:**

At $\kappa=0$, the MC smile (solid) sits below the Hagan smile (dashed) and the residual is mostly negative, reaching −250 bps in the right wing. Hagan
overshoots when $\nu$ is large.

As $\kappa$ increases, the solid lines move above the dashed lines and the residual flips to mostly positive (+10 to +30 bps), with a slight drop back to negative in the far right wing.

**What this tells us:**

$\kappa$ is not just equivalent to simply reducing $\nu$, if that were the case then the residual shape would stay the same, it would just shrink.
Instead, the residual changes sign and changes shape as κ increases. That means $\kappa$ introduces smile dynamics that no constant ($\nu$, $\rho$) pair can reproduce exactly. It is a genuine new degree of freedom.

Although, the effective parameter mapping still works well as residuals drop from ~250 bps at $\kappa=0$ to ~10–30 bps at $\kappa>=2$.

In [ ]:
# what happens when vol mean-reverts to a different level
# case 1 : $\theta$ > alpha
# case 2 : theta < alpha
kap_test = 2.0
theta_ratios = [0.5, 0.75, 1.0, 1.5, 2.0] #theta/alpha
colors_c = plt.cm.coolwarm(np.linspace(0.05, 0.95, len(theta_ratios)))

# Calibrate alpha with the standard mapping (theta = alpha)
an_ref = european_analytics.EuropeanAnalyticsMRSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_, kap_test)
alpha_cal = an_ref.alpha

# Log-symmetric strike grid
logm_max = 0.8
K_grid = F_s * np.exp(np.linspace(-logm_max, logm_max, 31))

# Hagan effective smile (assumes theta=alpha baseline)
hagan_vols = an_ref.implied_vol_smile(K_grid)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

rows = []
for ratio, col in zip(theta_ratios, colors_c):
    theta_val = ratio * alpha_cal

    sim = mSABR(F0=F_s, A0=alpha_cal, rho=rho, T=T,
                n_steps=N_STEPS, n_paths=N_PATHS,
                beta=beta, nu=nu, kappa=kap_test, theta=theta_val, seed=SEED)
    F_paths = sim.run()[0]
    F_shifted = F_paths[:, -1]
    F_T = F_shifted - lambda_

    mc_vols  = an_ref.implied_vol_smile_mc(F_shifted, K_grid)
    residual = (mc_vols - hagan_vols) * 1e4

    lab = f'theta/alpha={ratio:.2f}'

    axes[0].plot(K_grid, mc_vols, '-', color=col, lw=1.8, label=lab)
    axes[1].plot(K_grid, residual, '-', color=col, lw=1.8, label=lab)

    atm_mc = np.interp(F_s, K_grid, mc_vols)
    max_res = np.nanmax(np.abs(residual))
    rows.append((ratio, theta_val, atm_mc, max_res))

# Hagan baseline on left panel
axes[0].plot(K_grid, hagan_vols, 'k--', lw=1.5, label='Hagan (theta=alpha)')

axes[0].set_title('MC smile by theta/α ratio (solid) vs Hagan theta=α (dashed)')
axes[0].set_xlabel('Strike'); axes[0].set_ylabel('Implied vol')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Residual: MC − Hagan(theta=α)')
axes[1].set_xlabel('Strike'); axes[1].set_ylabel('Residual (bps)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

fig.suptitle(f'Experiment C: theta ≠ α  (κ={kap_test}, ν={nu}, β={beta}, ρ={rho})', y=1.02)
plt.tight_layout(); plt.show()

print(f"{'theta/α':>6}{'theta':>10}{'ATM_mc':>10}{'max_resid_bps':>15}")
for ratio, theta_val, atm, mr in rows:
    print(f"{ratio:6.2f}{theta_val:10.5f}{atm:10.5f}{mr:15.1f}")

### Experiment: what happens when vol mean-reverts to a different level

**Setup:** All prior experiments set $\theta = \alpha$ so that vol starts at its long-run mean, isolating the pure mean-reversion effect on smile shape. Here we break
that assumption: keep $\alpha$ (calibrated from $\sigma_{atm}$) but set $\theta$ to $0.5\alpha$,
$0.75\alpha$, $\alpha$, $1.5\alpha$, $2\alpha$. The Hagan effective formula still assumes $\theta = \alpha$.

**What we see:**

- $\theta/\alpha$ = 1.0 (gray): residual near zero — the baseline, confirming the effective mapping works when its assumption holds.
- $\theta/\alpha$ > 1 (red/orange): the MC smile sits well above the Hagan baseline. At $\theta/\alpha$ = 2.0, residuals reach ~1000 bps. Vol drifts upward from $\alpha$ toward the higher $\theta$, so realized volatility over the option life is higher than what Hagan assumed.
- $\theta/\alpha$ < 1 (blue): the MC smile drops below Hagan. At $\theta/\alpha$ = 0.5, residuals are ~−400 bps. Vol drifts downward, reducing realized volatility.

The effect is asymmetric: $\theta/\alpha$ = 2.0 produces larger residuals than $\theta/\alpha$ = 0.5

**Takeaway:** The effective parameter mapping assumes $\theta = \alpha$. When this assumption is violated, the Hagan formula gets the overall vol level wrong —
not just the smile shape. To handle $\theta \neq \alpha$ properly, we'd need to do something else.

In [ ]:
# Experiment B: does the quality of the effective parameter mapping depend on beta
# How well does the approximation hold across betas

betas_test = [0.0, 0.25, 0.5, 0.75, 1.0]
kappas_B   = [0.0, 4.0]
colors_beta = plt.cm.plasma(np.linspace(0.1, 0.9, len(betas_test)))

logm_max = 0.8
K_grid = F_s * np.exp(np.linspace(-logm_max, logm_max, 31))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for col_idx, kap in enumerate(kappas_B):
    ax_smile = axes[0, col_idx]
    ax_resid = axes[1, col_idx]
    rows = []

    for b, col in zip(betas_test, colors_beta):
        an = european_analytics.EuropeanAnalyticsMRSABR(
            F_s, T, b, nu, rho, sigma_atm, lambda_, kap)

        sim = mSABR(F0=F_s, A0=an.alpha, rho=rho, T=T,
                    n_steps=N_STEPS, n_paths=N_PATHS,
                    beta=b, nu=nu, kappa=kap, theta=an.alpha, seed=SEED)
        F_paths = sim.run()[0]
        F_shifted = F_paths[:, -1]

        mc_vols    = an.implied_vol_smile_mc(F_shifted, K_grid)
        hagan_vols = an.implied_vol_smile(K_grid)
        residual   = (mc_vols - hagan_vols) * 1e4

        lab = f'beta={b:.2f}'
        ax_smile.plot(K_grid, mc_vols, '-', color=col, lw=1.8, label=lab)
        ax_smile.plot(K_grid, hagan_vols, '--', color=col, lw=1, alpha=0.5)
        ax_resid.plot(K_grid, residual, '-', color=col, lw=1.8, label=lab)

        max_res = np.nanmax(np.abs(residual))
        rows.append((b, an.nu_eff, an.rho_eff, an.alpha, max_res))

    ax_smile.set_title(f'k={kap:.0f}: MC (solid) vs Hagan (dashed)')
    ax_smile.set_xlabel('Strike'); ax_smile.set_ylabel('Implied vol')
    ax_smile.legend(fontsize=8); ax_smile.grid(alpha=0.3)

    ax_resid.axhline(0, color='black', lw=0.8)
    ax_resid.set_title(f'k={kap:.0f}: Residual (MC − Hagan)')
    ax_resid.set_xlabel('Strike'); ax_resid.set_ylabel('Residual (bps)')
    ax_resid.legend(fontsize=8); ax_resid.grid(alpha=0.3)

    print(f"\nk = {kap:.0f}")
    print(f"{'beta':>6}{'nu_eff':>9}{'rho_eff':>9}{'alpha':>10}{'max_resid_bps':>15}")
    for b, ne, re, al, mr in rows:
        print(f"{b:6.2f}{ne:9.3f}{re:+9.3f}{al:10.5f}{mr:15.1f}")

fig.suptitle(f'Experiment: beta x kappa interaction  (nu={nu}, T={T}, rho={rho})', y=1.02)
plt.tight_layout(); plt.show()

### Experiment : Does the quality of the effective parameter mapping depend on beta 

**Setup:** Sweep $\beta$ = {0, 0.25, 0.5, 0.75, 1.0} at $\kappa=0$ (standard SABR) and $\kappa=4$ (mean-reverting). The effective parameter formulas in the paper do not
depend on beta, so $\nu_{eff}$ and $\rho_{eff}$ are identical across all $\beta$  values — only $\alpha$  changes (through the Hagan ATM inversion).

**What we see:**

At $\kappa=0$ (left), all $\beta$ values show the same pattern: large negative residuals (−50 to −200 bps), with Hagan overshooting the MC smile. The residual shape
is similar across $\beta$ and the Hagan approximation error is driven by large $\nu^2$ term, not by the choice of $\beta$. The boundary cases $\beta=0$ and $\beta=1$ show slightly worse
wing behavior than the intermediate values.

At $\kappa=4$ (right), residuals shrink dramatically to ±20 bps for most strikes. Mean reversion reduces $\nu_{eff}$, putting Hagan back in its accurate regime regardless of $\beta$. All the values of $\beta$  are mostly well behaved with low errors except at the extreme wings.

In [ ]:
# Experiment: divergence of effective Hagan's apprximation from the MC parameters
# max absolute residual (MC − Hagan) in bps across the strike grid.

kappas_D = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]
Ts_D     = [0.5, 1.0, 2.0, 5.0, 10.0]

logm_max = 0.6
K_grid = F_s * np.exp(np.linspace(-logm_max, logm_max, 21))

# Store results
max_resid = np.full((len(Ts_D), len(kappas_D)), np.nan)
nu_effs   = np.full((len(Ts_D), len(kappas_D)), np.nan)
atm_resid = np.full((len(Ts_D), len(kappas_D)), np.nan)

for i, T_val in enumerate(Ts_D):
    for j, kap in enumerate(kappas_D):
        an = european_analytics.EuropeanAnalyticsMRSABR(
            F_s, T_val, beta, nu, rho, sigma_atm, lambda_, kap)

        sim = mSABR(F0=F_s, A0=an.alpha, rho=rho, T=T_val,n_steps=max(int(T_val * 500), 200), n_paths=N_PATHS,beta=beta, nu=nu, kappa=kap, theta=an.alpha, seed=20)
        F_shifted = sim.run()[0][:, -1]

        mc_vols    = an.implied_vol_smile_mc(F_shifted, K_grid)
        hagan_vols = an.implied_vol_smile(K_grid)
        residual   = (mc_vols - hagan_vols) * 1e4

        max_resid[i, j] = np.nanmax(np.abs(residual))
        nu_effs[i, j]   = an.nu_eff
        atm_resid[i, j] = residual[len(K_grid)//2]

        print(f"  T={T_val:4.1f}, k={kap:4.1f}, kT={kap*T_val:5.1f}, "f"nu_eff={an.nu_eff:.3f}, max_resid={max_resid[i,j]:.0f} bps")

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# max absolute residual
im1 = axes[0].imshow(max_resid, aspect='auto', cmap='RdYlGn_r', origin='lower')
axes[0].set_xticks(range(len(kappas_D))); axes[0].set_xticklabels(kappas_D)
axes[0].set_yticks(range(len(Ts_D))); axes[0].set_yticklabels(Ts_D)
axes[0].set_xlabel('k'); axes[0].set_ylabel('T')
axes[0].set_title('Max |residual| (bps)')
for i in range(len(Ts_D)):
    for j in range(len(kappas_D)):
        axes[0].text(j, i, f'{max_resid[i,j]:.0f}', ha='center', va='center', fontsize=8)
plt.colorbar(im1, ax=axes[0])

# Right: ν_eff
im2 = axes[1].imshow(nu_effs, aspect='auto', cmap='viridis', origin='lower')
axes[1].set_xticks(range(len(kappas_D))); axes[1].set_xticklabels(kappas_D)
axes[1].set_yticks(range(len(Ts_D))); axes[1].set_yticklabels(Ts_D)
axes[1].set_xlabel('k'); axes[1].set_ylabel('T')
axes[1].set_title('nu_eff')
for i in range(len(Ts_D)):
    for j in range(len(kappas_D)):
        axes[1].text(j, i, f'{nu_effs[i,j]:.3f}', ha='center', va='center',
                     fontsize=7, color='white' if nu_effs[i,j] < 0.3 else 'black')
plt.colorbar(im2, ax=axes[1])

fig.suptitle(f'Experiment D: (k, T) divergence map  (nu={nu}, beta={beta}, rho={rho})', y=1.02)
plt.tight_layout(); plt.show()

### Experiment: Divergence of effective Hagan's approximation from the MC parameters, (k, T) Divergence heatmap

**Setup:** Sweep $\kappa$ = {0, 0.5, 1, 2, 4, 8} and T = {0.5, 1, 2, 5, 10}. For each ($\kappa$, T) pair, run MC and compare against Hagan with effective parameters.
The left heatmap shows max |residual| in bps, the right shows $\nu_{eff}$.

**What we see:**

The $\kappa=0$ column (standard SABR, no mean reversion) gets progressively worse as T increases: 108 -> 143 -> 73 -> 193 -> 440 bps. This is the Hagan formula
breaking down at large T — with $\nu$=0.5.

Moving right across any row (increasing $\kappa$), residuals drop sharply. For example at T=10: 440 -> 82 -> 69 -> 42 -> 23 -> 15 bps. Mean reversion shrinks
$\nu_{eff}$ (right heatmap), bringing $\nu^2T$ down and back into the regime where Hagan is accurate.

**Conclusion:** For a given maturity T, even modest mean reversion (kT ≥ 2) dramatically improves the Hagan approximation by suppressing the effective vol-of-vol. This is why the mrSABR model is particularly useful for long-dated derivatives as it tames the $\nu^2T$ blow-up that makes standard SABR unreliable beyond a few years.